# SPEAR-Net — LAMES training (Colab, T4)

**SPEAR-Net**: a lightweight, color-prior-guided, recall-optimized network for
fine-grained segmentation of mining structures, with emphasis on artisanal &
small-scale (illegal-prone) mining.

This notebook runs the **whole pipeline** end-to-end on a free Colab **T4 GPU**:
clone repo → install deps → load LAMES → visualize the Color-Spectral Prior → build
SPEAR-Net → report efficiency → **train** → evaluate (per-class + area-stratified
recall) → CSP explainability overlays.

> **Runtime → Change runtime type → T4 GPU**, then *Runtime → Run all*.

If you hit a bug, just re-pull the repo (cell 2) — the maintainer updates this notebook
and the source in place.


## 1. Check GPU

In [ ]:
!nvidia-smi || echo "No GPU — set Runtime > Change runtime type > T4 GPU"
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())


## 2. Clone the repository

Re-run this cell to pull the latest fixes.

In [ ]:
import os, subprocess

REPO_URL = "https://github.com/prakhar443/illegal_mining.git"
BRANCH   = "claude/busy-euler-kswww1"
REPO_DIR = "illegal_mining"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=False)
    # Fallback if the branch isn't the default and clone -b failed
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=False)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=False)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=False)

%cd {REPO_DIR}
!git log --oneline -1


## 3. Install dependencies

~2-3 min on a fresh Colab. `datasets`/`timm`/`smp` are the key ones.

In [ ]:
!pip install -q "timm>=0.9.12" "segmentation-models-pytorch>=0.3.3" \
    "datasets>=2.16" "huggingface-hub>=0.20" ptflops scikit-learn pyyaml
!pip install -q -e .
print("Done. If imports fail below, Runtime > Restart session, then re-run from cell 3.")


## 4. Imports & config

We use a small **dev subset** so a run finishes in minutes. For the paper, set `subset_train = None` for the full 168k patches.

In [ ]:
import sys; sys.path.insert(0, "src")
import torch
from spearnet.config import load_config
from spearnet.data import build_dataloaders, compute_csp_priors
from spearnet.models import build_model
from spearnet.engine import Trainer, evaluate
from spearnet.utils import set_seed, measure_efficiency, count_parameters, save_prediction_panel

# ---- choose your experiment ----
CONFIG = "configs/spearnet_10class.yaml"   # or spearnet_3class.yaml / spearnet_binary.yaml
cfg = load_config(CONFIG)

# Colab-friendly dev settings (comment out for full runs)
cfg.data.subset_train = 1500
cfg.data.subset_val   = 400
cfg.optim.epochs      = 8
cfg.data.num_workers  = 2
cfg.run.device        = "cuda" if torch.cuda.is_available() else "cpu"

set_seed(cfg.run.seed)
print(f"Task={cfg.data.task} ({cfg.num_classes} classes) | csp={cfg.model.csp_mode} | "
      f"backbone={cfg.model.backbone} | device={cfg.run.device}")


## 5. Load LAMES from Hugging Face

First call downloads/caches the dataset split. Set `cfg.data.streaming = True` to avoid the full download.

In [ ]:
loaders = build_dataloaders(cfg, splits=("train", "val"))
print("train batches:", len(loaders["train"]), "| val batches:", len(loaders["val"]))
batch = next(iter(loaders["train"]))
print({k: tuple(v.shape) for k, v in batch.items()})
print("mask classes present in batch:", torch.unique(batch["mask"]).tolist())


## 6. Visualize a sample + the Color-Spectral Prior

ExG (greenness), Brightness, Redness (iron-oxide), RGB-VI — computed on the fly.

In [ ]:
import matplotlib.pyplot as plt
from spearnet.utils.viz import colorize_mask
from spearnet.data.priors import CSP_PRIOR_NAMES

img = batch["image"][0]
priors = compute_csp_priors(img.unsqueeze(0))[0]

fig, ax = plt.subplots(1, 6, figsize=(22, 4))
ax[0].imshow(img.permute(1,2,0).numpy()); ax[0].set_title("RGB")
ax[1].imshow(colorize_mask(batch["mask"][0].numpy())); ax[1].set_title("Mask")
for i, name in enumerate(CSP_PRIOR_NAMES):
    ax[i+2].imshow(priors[i].numpy(), cmap="viridis"); ax[i+2].set_title(name)
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()


## 7. Build SPEAR-Net + efficiency report

Params / GFLOPs / latency / peak VRAM back the deployability claims (≤ ~6 M params, sub-second T4 inference).

In [ ]:
model = build_model(cfg)
print("Parameters:", count_parameters(model))
stats = measure_efficiency(model, image_size=cfg.data.image_size, device=cfg.run.device)
import json; print(json.dumps(stats, indent=2))


## 8. Train

Logs per-class IoU each epoch and checkpoints the best model to `runs/`.

In [ ]:
trainer = Trainer(model, loaders, cfg)
summary = trainer.train()
print("best", cfg.run.save_best_metric, "=", summary["best_metric"])


## 9. Evaluate (per-class IoU + area-stratified recall)

In [ ]:
device = torch.device(cfg.run.device)
results = evaluate(model, loaders["val"], cfg, device, compute_area_stratified=True)
print(f"mIoU={results['miou']:.4f}  mF1={results['mean_f1']:.4f}  "
      f"mRecall={results['mean_recall']:.4f}  pixAcc={results['pixel_acc']:.4f}")
print("\nPer-class IoU / recall:")
for cls, m in results["per_class"].items():
    print(f"  {cls:>18}: IoU={m['iou']:.3f}  recall={m['recall']:.3f}  support={m['support']}")
print("\nArea-stratified recall:", results.get("area_stratified_recall"))


## 10. Explainability — predictions + CSP attention overlay

In [ ]:
model.eval()
with torch.no_grad():
    vb = next(iter(loaders["val"]))
    out = model(vb["image"].to(device))
    preds = out["logits"].argmax(1).cpu()
    attn = out.get("attn")

import os; os.makedirs("figures", exist_ok=True)
for i in range(min(3, preds.shape[0])):
    a = attn[i].cpu() if attn is not None else None
    save_prediction_panel(vb["image"][i], vb["mask"][i], preds[i],
                          f"figures/panel_{i}.png", attn=a, class_names=cfg.class_names)

from IPython.display import Image as IPImage, display
for i in range(min(3, preds.shape[0])):
    display(IPImage(f"figures/panel_{i}.png"))


## 11. Next steps — ablations, baselines & the full run

```python
# Ablation table (run each, compare runs/*/metrics_val.json)
for c in ["configs/ablation_1_baseline.yaml",
          "configs/ablation_2_recall_loss.yaml",
          "configs/ablation_3_csp_concat.yaml",
          "configs/ablation_4_full_spearnet.yaml"]:
    !python scripts/train.py --config {c} --set optim.epochs=8 data.subset_train=1500

# Baselines (U-Net / Attention U-Net / DeepLabV3+ / U-Net++)
!python scripts/train.py --config configs/baseline_unet.yaml --set model.name=deeplabv3p

# Full paper run (long): set subset to null and epochs to 40
!python scripts/train.py --config configs/spearnet_10class.yaml \
    --set data.subset_train=null optim.epochs=40
```

Tasks: switch `CONFIG` to `configs/spearnet_3class.yaml` for the **ASM-vs-LSM** headline
result, or `configs/spearnet_binary.yaml` for the mining detector.
